In [ ]:
import os
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ==============================
# CONFIG — CHEMINS CORRECTS
# ==============================
W1 = r"C:\Users\INFOTEC\OneDrive\Bureau\Pre_w1w2\Cross_Week_Results\Week1_Final.xlsx"
W2 = r"C:\Users\INFOTEC\OneDrive\Bureau\Pre_w2w3\Cross_Week_Results\Week2_Final.xlsx"
W3 = r"C:\Users\INFOTEC\OneDrive\Bureau\pre_w3w4\Cross_Week_Results\Week3_Final.xlsx"
OUT_DIR = r"C:\Users\INFOTEC\OneDrive\Bureau\newton_week_to_days"

os.makedirs(OUT_DIR, exist_ok=True)

PAYS_LIST = ['Cyclam', 'Germany', 'India', 'Korea', 'Kunshan', 'Tianjin', 'USA', 'SAME', 'SCEET']

# ==============================
# COULEURS
# ==============================
C_HDR_BG     = '1F4E79'
C_HDR_FG     = 'FFFFFF'
C_SUM_BG     = '1A5276'
C_ROW_ALT    = ['EBF5FB', 'FDFEFE']
C_MISSING    = 'E8E8E8'   # Gris = produit absent dans cette semaine (qty=0)
C_PERTURB_OK = 'C6EFCE'   # Vert  = perturbation <= 2%
C_PERTURB_KO = 'FFC7CE'   # Rouge = perturbation >  2%
C_DAY_HDRS   = ['BDD7EE', 'FCE4D6', 'E2EFDA', 'FFF2CC', 'F4CCFF', 'D9EAD3']
WK_COLORS    = ['1A6BAD', '1A8A6B', '8A1A6B']
DAYS         = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi']

thin = Side(style='thin', color='BFBFBF')
BORD = Border(left=thin, right=thin, top=thin, bottom=thin)


# ==============================
# HELPERS
# ==============================
def sc(cell, bold=False, bg=None, fg='000000', align='center', wrap=False, fmt=None):
    cell.font      = Font(name='Arial', bold=bold, color=fg, size=9)
    if bg:
        cell.fill  = PatternFill('solid', start_color=bg)
    cell.alignment = Alignment(horizontal=align, vertical='center', wrap_text=wrap)
    cell.border    = BORD
    if fmt:
        cell.number_format = fmt


def to_f(v):
    try:
        return float(str(v).replace(',', '.').replace(' ', ''))
    except:
        return 0.0


# ==============================
# NEWTON — stock quotidien
# ==============================
def newton_daily_usage(inv1, wu1, inv2, wu2, inv3, wu3):
    """
    Calcule l'usage quotidien sur 6 jours par interpolation de Newton.
    Noeuds : (semaine1, inv1), (semaine2, inv2), (semaine3, inv3)
    Perturb% = (Usage_Jd - WU/6) / (WU/6) x 100
    """
    results = []
    for inv_start, wu in [(inv1, wu1), (inv2, wu2), (inv3, wu3)]:
        xs = np.linspace(1, 3, 6)
        d1_n  = (1.01 - 0.99)
        d2_n  = (0.99 - 1.01)
        d12_n = (d2_n - d1_n) / 2
        factors = 0.99 + d1_n * (xs - 1) + d12_n * (xs - 1) * (xs - 2)

        raw_usage = (wu / 6) * factors
        if raw_usage.sum() > 0:
            raw_usage = raw_usage * (wu / raw_usage.sum())

        stock_avant = []
        stock_apres = []
        s = inv_start
        for u in raw_usage:
            stock_avant.append(round(s, 4))
            s = max(0.0, s - u)
            stock_apres.append(round(s, 4))

        results.append({
            'stock_avant': stock_avant,
            'usage':       [round(a - b, 4) for a, b in zip(stock_avant, stock_apres)],
            'stock_apres': stock_apres,
            'wu':          wu,
            'inv_start':   inv_start,
        })
    return results


# ==============================
# CHARGEMENT — union des PNs
# ==============================
def load_pays(pays):
    """
    Charge les 3 semaines pour un pays.
    UNION complete des Part Numbers :
      - Ordre : PNs de W1 d'abord, puis nouveaux de W2, puis nouveaux de W3
      - PN absent d'une semaine => ligne ajoutee avec qty = 0 (fond gris)
    """
    NUM_COLS = ['Real Inventory (Qty)', 'Weekly Usage (Qty)',
                'Unit Price (€)', 'Stock Value (€)', 'Weekly Target']
    raws = []
    for wpath in [W1, W2, W3]:
        xl = pd.read_excel(wpath, sheet_name=pays, header=0)
        for c in NUM_COLS:
            if c in xl.columns:
                xl[c] = xl[c].apply(to_f)
        xl['Part Number'] = xl['Part Number'].astype(str).str.strip()
        raws.append(xl)

    # Union ordonnee des PNs
    seen = {}
    for df in raws:
        for _, row in df.iterrows():
            pn = str(row['Part Number']).strip()
            if pn not in seen:
                seen[pn] = {
                    'Description':    row.get('Description', ''),
                    'Supplier':       row.get('Supplier', ''),
                    'Unit Price (€)': to_f(row.get('Unit Price (€)', 0)),
                }
    all_pns_ordered = list(seen.keys())

    # Reconstruire chaque df avec tous les PNs (0 si absent)
    dfs = []
    for df in raws:
        rows = []
        for pn in all_pns_ordered:
            match = df[df['Part Number'].astype(str).str.strip() == pn]
            if not match.empty:
                r = match.iloc[0].to_dict()
                r['_missing'] = False
                rows.append(r)
            else:
                rows.append({
                    'Part Number':          pn,
                    'Description':          seen[pn]['Description'],
                    'Supplier':             seen[pn]['Supplier'],
                    'Unit Price (€)':       seen[pn]['Unit Price (€)'],
                    'Real Inventory (Qty)': 0.0,
                    'Weekly Usage (Qty)':   0.0,
                    'Stock Value (€)':      0.0,
                    'Weekly Target':        0.0,
                    '_missing':             True,
                })
        dfs.append(pd.DataFrame(rows).reset_index(drop=True))

    return dfs


# ==============================
# SHEET WEEK
# ==============================
def write_week_sheet(wb, pays, week_num, df_wk, all_newton):
    ws = wb.create_sheet(f'📅 Week{week_num}')
    ws.freeze_panes = 'E4'
    NCOLS = 38

    # Ligne 1 : Titre
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=NCOLS)
    t = ws.cell(1, 1,
                f"📅  {pays} — Week {week_num}  |  Stock quotidien Newton  |  "
                f"Perturb% = (Usage_Jd - WU/6) / (WU/6)  |  "
                f"Gris = produit absent (qty=0)")
    sc(t, bold=True, bg=C_HDR_BG, fg=C_HDR_FG, align='left')
    ws.row_dimensions[1].height = 20

    # Ligne 2 : En-tetes groupes
    for c, lbl in zip([1, 2, 3, 4], ['Part Number', 'Description', 'Supplier', 'Unit Price (€)']):
        ws.merge_cells(start_row=2, start_column=c, end_row=3, end_column=c)
        sc(ws.cell(2, c, lbl), bold=True, bg=C_HDR_BG, fg=C_HDR_FG, wrap=True)

    ws.merge_cells(start_row=2, start_column=5, end_row=2, end_column=8)
    sc(ws.cell(2, 5, f'📌 Résumé Semaine {week_num}'), bold=True, bg='2E75B6', fg='FFFFFF')
    for c in [6, 7, 8]:
        sc(ws.cell(2, c), bg='2E75B6', fg='FFFFFF')

    for d in range(6):
        start_c = 9 + d * 5
        ws.merge_cells(start_row=2, start_column=start_c, end_row=2, end_column=start_c + 4)
        day_cell = ws.cell(2, start_c, f'J{d+1} — {DAYS[d]}')
        day_cell.font      = Font(name='Arial', bold=True, color='1F3864', size=9)
        day_cell.fill      = PatternFill('solid', start_color=C_DAY_HDRS[d])
        day_cell.alignment = Alignment(horizontal='center', vertical='center')
        day_cell.border    = BORD
        for c in range(start_c + 1, start_c + 5):
            cell = ws.cell(2, c)
            cell.fill   = PatternFill('solid', start_color=C_DAY_HDRS[d])
            cell.border = BORD
    ws.row_dimensions[2].height = 18

    # Ligne 3 : Sous-en-tetes
    for i, lbl in enumerate(['Inv. Réel', 'Usage Hebdo', 'Val. Stock', 'Cible']):
        cell = ws.cell(3, 5 + i, lbl)
        cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
        cell.fill      = PatternFill('solid', start_color='1A5276')
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border    = BORD

    for d in range(6):
        for j, lbl in enumerate(['Avant', 'Usage', 'Après', 'Val.', 'Perturb%']):
            cell = ws.cell(3, 9 + d * 5 + j, lbl)
            cell.font      = Font(name='Arial', bold=True, color='1F3864', size=9)
            cell.fill      = PatternFill('solid', start_color=C_DAY_HDRS[d])
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border    = BORD
    ws.row_dimensions[3].height = 28

    # Lignes de donnees
    for r_idx, row in df_wk.iterrows():
        pn         = str(row['Part Number']).strip()
        er         = r_idx + 4
        is_missing = row.get('_missing', False) == True
        bg         = C_MISSING if is_missing else C_ROW_ALT[r_idx % 2]
        up         = to_f(row.get('Unit Price (€)', 0))
        ri         = to_f(row.get('Real Inventory (Qty)', 0))
        wu         = to_f(row.get('Weekly Usage (Qty)', 0))
        sv         = to_f(row.get('Stock Value (€)', 0))
        wt         = to_f(row.get('Weekly Target', 0))

        def wc(col, val, fmt=None, bg_ov=None):
            cell = ws.cell(er, col, val)
            cell.font      = Font(name='Arial', size=9)
            cell.fill      = PatternFill('solid', start_color=bg_ov or bg)
            cell.alignment = Alignment(
                horizontal='right' if isinstance(val, (int, float)) else 'left',
                vertical='center')
            cell.border = BORD
            if fmt:
                cell.number_format = fmt

        wc(1, pn)
        wc(2, str(row.get('Description', '')) if pd.notna(row.get('Description', '')) else '')
        wc(3, str(row.get('Supplier', ''))    if pd.notna(row.get('Supplier', ''))    else '')
        wc(4, up, '#,##0.0000')
        wc(5, ri, '#,##0.00')
        wc(6, wu, '#,##0.00')
        wc(7, sv, '#,##0.00')
        wc(8, wt, '#,##0.00')

        nt  = all_newton.get(pn)
        wu6 = wu / 6 if wu != 0 else 0

        for d in range(6):
            base_c = 9 + d * 5
            if nt:
                wk_data = nt[week_num - 1]
                avant   = wk_data['stock_avant'][d]
                usage   = wk_data['usage'][d]
                apres   = wk_data['stock_apres'][d]
                sv_d    = round(apres * up, 4)
            else:
                avant = usage = apres = sv_d = 0.0

            perturb = ((usage - wu6) / wu6 * 100) if wu6 != 0 else 0.0

            wc(base_c,     avant, '#,##0.00')
            wc(base_c + 1, usage, '#,##0.00')
            wc(base_c + 2, apres, '#,##0.00')
            wc(base_c + 3, sv_d,  '#,##0.00')

            p_cell = ws.cell(er, base_c + 4, round(perturb, 2))
            p_cell.font      = Font(name='Arial', size=9)
            p_cell.fill      = PatternFill('solid',
                                           start_color=C_PERTURB_OK if abs(perturb) <= 2 else C_PERTURB_KO)
            p_cell.alignment = Alignment(horizontal='right', vertical='center')
            p_cell.border    = BORD
            p_cell.number_format = '0.00"%"'

    ws.column_dimensions['A'].width = 14
    ws.column_dimensions['B'].width = 28
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 11
    for c in range(5, NCOLS + 1):
        ws.column_dimensions[get_column_letter(c)].width = 11


# ==============================
# SHEET SUMMARY
# ==============================
def write_summary(wb, pays, dfs, all_newton):
    ws = wb.create_sheet('📋 Summary', 0)
    ws.freeze_panes = 'E4'
    NCOLS = 16

    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=NCOLS)
    t = ws.cell(1, 1,
                f'{pays} — Summary 3 Semaines  |  '
                f'Inv. Réel · Usage · Val. Stock · Perturbation% Moyenne  |  '
                f'Gris = produit absent (qty=0)')
    t.font      = Font(name='Arial', bold=True, color='FFFFFF', size=10)
    t.fill      = PatternFill('solid', start_color=C_HDR_BG)
    t.alignment = Alignment(horizontal='left', vertical='center')
    t.border    = BORD
    ws.row_dimensions[1].height = 20

    for c, lbl in zip([1, 2, 3, 4], ['Part Number', 'Description', 'Supplier', 'Unit Price (€)']):
        ws.merge_cells(start_row=2, start_column=c, end_row=3, end_column=c)
        cell = ws.cell(2, c, lbl)
        cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
        cell.fill      = PatternFill('solid', start_color=C_HDR_BG)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border    = BORD

    for w in range(3):
        start_c = 5 + w * 4
        ws.merge_cells(start_row=2, start_column=start_c, end_row=2, end_column=start_c + 3)
        cell = ws.cell(2, start_c, f'Week {w+1}')
        cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
        cell.fill      = PatternFill('solid', start_color=WK_COLORS[w])
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border    = BORD
        for c in range(start_c + 1, start_c + 4):
            cell = ws.cell(2, c)
            cell.fill   = PatternFill('solid', start_color=WK_COLORS[w])
            cell.border = BORD
    ws.row_dimensions[2].height = 18

    for w in range(3):
        for j, lbl in enumerate(['Inv. Réel', 'Usage', 'Val. Stock', 'Perturb% moy.']):
            cell = ws.cell(3, 5 + w * 4 + j, lbl)
            cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
            cell.fill      = PatternFill('solid', start_color=WK_COLORS[w])
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border    = BORD
    ws.row_dimensions[3].height = 30

    all_pns = list(dfs[0]['Part Number'].astype(str).str.strip())

    for r_idx, pn in enumerate(all_pns):
        er   = r_idx + 4
        row1 = dfs[0][dfs[0]['Part Number'].astype(str).str.strip() == pn]

        is_missing_all = all(
            dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn].iloc[0].get('_missing', False) == True
            if not dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn].empty else True
            for w in range(3)
        )
        bg = C_MISSING if is_missing_all else C_ROW_ALT[r_idx % 2]

        def wc(col, val, fmt=None, bg_ov=None):
            cell = ws.cell(er, col, val)
            cell.font      = Font(name='Arial', size=9)
            cell.fill      = PatternFill('solid', start_color=bg_ov or bg)
            cell.alignment = Alignment(
                horizontal='right' if isinstance(val, (int, float)) else 'left',
                vertical='center')
            cell.border = BORD
            if fmt:
                cell.number_format = fmt

        if not row1.empty:
            r = row1.iloc[0]
            wc(1, pn)
            wc(2, str(r.get('Description', '')) if pd.notna(r.get('Description', '')) else '')
            wc(3, str(r.get('Supplier', ''))    if pd.notna(r.get('Supplier', ''))    else '')
            wc(4, to_f(r.get('Unit Price (€)', 0)), '#,##0.0000')

        nt = all_newton.get(pn)
        for w in range(3):
            base_c = 5 + w * 4
            row_w  = dfs[w][dfs[w]['Part Number'].astype(str).str.strip() == pn]
            if not row_w.empty:
                r   = row_w.iloc[0]
                ri  = to_f(r.get('Real Inventory (Qty)', 0))
                wu  = to_f(r.get('Weekly Usage (Qty)', 0))
                sv  = to_f(r.get('Stock Value (€)', 0))
                wu6 = wu / 6 if wu != 0 else 0
                if nt and wu6 != 0:
                    usages   = nt[w]['usage']
                    perturbs = [(u - wu6) / wu6 * 100 for u in usages]
                    pb = round(np.mean(perturbs), 2)
                else:
                    pb = 0.0
                wc(base_c,     ri, '#,##0.00')
                wc(base_c + 1, wu, '#,##0.00')
                wc(base_c + 2, sv, '#,##0.00')
                p_cell = ws.cell(er, base_c + 3, pb)
                p_cell.font      = Font(name='Arial', size=9)
                p_cell.fill      = PatternFill('solid',
                                               start_color=C_PERTURB_OK if abs(pb) <= 2 else C_PERTURB_KO)
                p_cell.alignment = Alignment(horizontal='right', vertical='center')
                p_cell.border    = BORD
                p_cell.number_format = '0.00"%"'
            else:
                for c in range(base_c, base_c + 4):
                    wc(c, '-')

    # Ligne TOTAL / MOYENNE
    tot_row = len(all_pns) + 4
    ws.merge_cells(start_row=tot_row, start_column=1, end_row=tot_row, end_column=4)
    cell = ws.cell(tot_row, 1, 'TOTAL / MOYENNE')
    cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
    cell.fill      = PatternFill('solid', start_color=C_SUM_BG)
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.border    = BORD

    for w in range(3):
        for j in range(4):
            c     = 5 + w * 4 + j
            col_l = get_column_letter(c)
            formula = (f'=AVERAGE({col_l}4:{col_l}{tot_row-1})'
                       if j == 3 else
                       f'=SUM({col_l}4:{col_l}{tot_row-1})')
            fmt   = '0.00"%"' if j == 3 else '#,##0.00'
            cell  = ws.cell(tot_row, c, formula)
            cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
            cell.fill      = PatternFill('solid', start_color=C_SUM_BG)
            cell.alignment = Alignment(horizontal='right', vertical='center')
            cell.border    = BORD
            cell.number_format = fmt

    ws.column_dimensions['A'].width = 14
    ws.column_dimensions['B'].width = 28
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 11
    for c in range(5, NCOLS + 1):
        ws.column_dimensions[get_column_letter(c)].width = 13


# ==============================
# MAIN
# ==============================
def main():
    for pays in PAYS_LIST:
        print(f'⏳ Processing {pays}...')
        try:
            dfs = load_pays(pays)
        except Exception as e:
            print(f'  ⚠️  Skipped {pays}: {e}')
            continue

        df1, df2, df3 = dfs

        all_newton = {}
        all_pns    = list(df1['Part Number'].astype(str).str.strip())

        for pn in all_pns:
            def get_val(df, pn_=pn, col=None):
                r = df[df['Part Number'].astype(str).str.strip() == pn_]
                return to_f(r.iloc[0][col]) if not r.empty else 0.0

            inv1 = get_val(df1, col='Real Inventory (Qty)')
            wu1  = get_val(df1, col='Weekly Usage (Qty)')
            inv2 = get_val(df2, col='Real Inventory (Qty)')
            wu2  = get_val(df2, col='Weekly Usage (Qty)')
            inv3 = get_val(df3, col='Real Inventory (Qty)')
            wu3  = get_val(df3, col='Weekly Usage (Qty)')

            all_newton[pn] = newton_daily_usage(inv1, wu1, inv2, wu2, inv3, wu3)

        wb = Workbook()
        wb.remove(wb.active)

        write_summary(wb, pays, dfs, all_newton)
        write_week_sheet(wb, pays, 1, df1, all_newton)
        write_week_sheet(wb, pays, 2, df2, all_newton)
        write_week_sheet(wb, pays, 3, df3, all_newton)

        out_path = os.path.join(OUT_DIR, f'{pays}_Stock_Quotidien_Full.xlsx')
        wb.save(out_path)
        print(f'  ✅ Saved: {out_path}')

    print('\n🎯 TERMINÉ — Tous les pays générés!')


main()

⏳ Processing Cyclam...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton\Cyclam_Stock_Quotidien_Full.xlsx
⏳ Processing Germany...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton\Germany_Stock_Quotidien_Full.xlsx
⏳ Processing India...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton\India_Stock_Quotidien_Full.xlsx
⏳ Processing Korea...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton\Korea_Stock_Quotidien_Full.xlsx
⏳ Processing Kunshan...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton\Kunshan_Stock_Quotidien_Full.xlsx
⏳ Processing Tianjin...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton\Tianjin_Stock_Quotidien_Full.xlsx
⏳ Processing USA...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton\USA_Stock_Quotidien_Full.xlsx
⏳ Processing SAME...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton\SAME_Stock_Quotidien_Full.xlsx
⏳ Processing SCEET...
  ✅ Saved: C:\Users\INFOTEC\OneDrive\Bureau\newton\SCEET_Stock_Quotidien_Full.xlsx

🎯 TERMINÉ — Tous les pays générés!
